# Inverse model: product to predominant profile

This notebook reuses the M2 propensity model, `P(click | user, product)`, in reverse: given a product, it estimates which user profiles present the highest probability of a click.

The procedure fixes the characteristics of a product, sweeps through the combinations of user profile (age, sex, income, car, municipality) and predicts `P(click)` for each one. The marginal probability of a value is the mean `P(click)` of that value averaged over all the combinations that contain it, and it makes it possible to identify which level stands out for each product.

Given that the user variables are synthetic, their effect on `P(click)` is small and the prediction depends mainly on the product. The aim here is the method, not the magnitude of the demographic effect.

## 0 · Libraries and paths

In [ ]:
import warnings
import sys
from itertools import product as combinaciones_de
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Add src/ to the path and reuse the shared helper (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

PROCESSED_PATH = find_project_root() / "data" / "processed"
EXTERNAL_PATH = find_project_root() / "data" / "external_clean"
np.random.seed(42)
print("Carpeta de datos:", PROCESSED_PATH)

## 1 · Data and variables (user + product + embeddings)

In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})
events = pd.read_csv(PROCESSED_PATH / "events.csv")
products = pd.read_csv(PROCESSED_PATH / "products.csv")
tabla_marcas = pd.read_csv(EXTERNAL_PATH / "tabla_marcas.csv", sep=None, engine="python")
demo_segments = pd.read_csv(PROCESSED_PATH / "users_demo_segments.csv")[["id_user", "demo_cluster"]]

events = events[events["event_type"].isin(["click", "open"])].copy()
events["target"] = (events["event_type"] == "click").astype(int)

In [ ]:
# --- user variables (same criterion as notebook 07) ---
u = users.copy()
u["gender_enc"] = (u["gender"] == "H").astype(int)
u["labor_status_enc"] = u["labor_status"].map({"employed": 2, "unemployed": 1, "inactive": 0}).fillna(0).astype(int)
u["civil_status_enc"] = u["civil_status"].map({"casado": 3, "soltero": 0, "divorciado": 1, "viudo": 2}).fillna(0).astype(int)
u["tiene_coche_enc"] = u["tiene_coche"].astype(int)
u["num_room_enc"] = u["num_room"].map({"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}).fillna(2).astype(int)


def tamano_hogar(texto):
    if pd.isna(texto):
        return 3
    texto = str(texto).strip()
    if texto[:1].isdigit() and texto[0] != "0":
        return int(texto[0])
    return 5


u["size_hogar_enc"] = u["size_hogar"].apply(tamano_hogar)


def edad_a_grupo(edad):
    if pd.isna(edad):
        return -1
    if edad < 25:
        return 0
    elif edad < 35:
        return 1
    elif edad < 45:
        return 2
    elif edad < 55:
        return 3
    elif edad < 65:
        return 4
    else:
        return 5


u["age_cat"] = u["age"].apply(edad_a_grupo).astype(int)
u = u.merge(demo_segments, on="id_user", how="left")
u["demo_cluster"] = u["demo_cluster"].fillna(-1).astype(int)

USER_FEATS = ["age_cat", "gender_enc", "labor_status_enc", "civil_status_enc", "tiene_coche_enc",
              "size_hogar_enc", "num_room_enc", "ipa_class", "mun_type", "distance_type", "demo_cluster"]

In [ ]:
# --- product variables (sector marketing attributes + cpl + embeddings) ---
posibles_attrs = ["Urgencia", "Racionalidad", "RiesgoPercibido", "CicloDecision",
                  "Implicación", "Necesidad", "SensibilidadPrecio", "CompetenciaAlta", "PrecioMedio"]
attr_cols = [c for c in posibles_attrs if c in tabla_marcas.columns]
texto_a_numero = {"muy baja": 0, "baja": 1, "baja-media": 1, "media": 2, "medio": 2, "media-alta": 3,
                  "alta": 4, "muy alta": 5, "alto": 4, "bajo": 1, "no": 0, "si": 1, "sí": 1,
                  "corto": 0, "medio-largo": 2, "largo": 3}
tm = tabla_marcas.copy()
attr_num_cols = []
for col in attr_cols:
    nueva = col + "_num"
    if tm[col].dtype == object:
        tm[nueva] = tm[col].astype(str).str.strip().str.lower().map(texto_a_numero)
    else:
        tm[nueva] = pd.to_numeric(tm[col], errors="coerce")
    attr_num_cols.append(nueva)
col_sector = [c for c in tabla_marcas.columns if c.lower() == "sector"][0]
tm["sector_norm"] = tm[col_sector].astype(str).str.strip().str.lower()
sector_attrs = tm.groupby("sector_norm")[attr_num_cols].mean().reset_index()

p = products.copy()
p["sector_norm"] = p["sector"].astype(str).str.strip().str.lower()
p["cpl"] = pd.to_numeric(p["cpl"], errors="coerce")
p["sector_code"] = p["sector"].astype("category").cat.codes
p["prod_cat_code"] = p["product_new"].astype("category").cat.codes
p = p.merge(sector_attrs, on="sector_norm", how="left")

emb_df = pd.read_csv(PROCESSED_PATH / "subject_emb.csv")
EMB_COLS = [c for c in emb_df.columns if c.startswith("emb_")]
p = p.merge(emb_df, on="id_product", how="left")

PROD_NUM_FEATS = ["cpl"] + attr_num_cols
PROD_FEATS = ["sector_code", "prod_cat_code"] + PROD_NUM_FEATS + EMB_COLS

# events table
df = events[["id_user", "id_product", "target"]].copy()
df = df.merge(u[["id_user"] + USER_FEATS], on="id_user", how="left")
df = df.merge(p[["id_product", "product_new"] + PROD_FEATS], on="id_product", how="left")
df = df.dropna(subset=["product_new"]).reset_index(drop=True)
for col in PROD_NUM_FEATS + EMB_COLS:
    df[col] = df[col].fillna(df[col].median())
print("Tabla de eventos:", df.shape)

## 2 · We train the forward model `P(click | user, product)`

This is the same model as M2 (Variant B). We will use it in inverse mode in the next section.

In [ ]:
FEATS = USER_FEATS + PROD_FEATS
CAT_COLS = ["demo_cluster", "sector_code", "prod_cat_code"]
cat_idx = [FEATS.index(c) for c in CAT_COLS]

modelo = lgb.LGBMClassifier(objective="binary", n_estimators=300, learning_rate=0.05, num_leaves=15,
                            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                            random_state=42, verbose=-1, n_jobs=-1)
modelo.fit(df[FEATS].values.astype(float), df["target"].values, categorical_feature=cat_idx)

# Mean characteristics of each product category (so we can "fix" a product)
features_por_categoria = (p.dropna(subset=["product_new"]).groupby("product_new")[PROD_FEATS]
                          .agg({**{c: "mean" for c in PROD_NUM_FEATS + EMB_COLS},
                                "sector_code": "first", "prod_cat_code": "first"}))

# REFERENCE user profile = the most frequent value of each variable
referencia = {}
for col in USER_FEATS:
    referencia[col] = int(df[col].mode().iloc[0])
print("Modelo entrenado. Perfil de referencia:", referencia)

## 3 · Inverse query: product → profiles with the highest click probability

For a product, we test every combination of **age × sex × income × car × municipality**
(the other variables stay at their reference value) and predict `P(click)`.

In [ ]:
GRUPOS_EDAD = {0: "18-24", 1: "25-34", 2: "35-44", 3: "45-54", 4: "55-64", 5: "65+"}

# Variables we are going to sweep and their possible values
VARIABLES_BARRIDAS = {
    "age_cat": [0, 1, 2, 3, 4, 5],
    "gender_enc": [0, 1],
    "ipa_class": [0, 1, 2, 3, 4],
    "tiene_coche_enc": [0, 1],
    "mun_type": [0, 1, 2, 3, 4, 5],
}
nombres_barrido = list(VARIABLES_BARRIDAS)
todas_las_combinaciones = list(combinaciones_de(*VARIABLES_BARRIDAS.values()))
print("Nº de combinaciones de perfil por producto:", len(todas_las_combinaciones))


def consulta_inversa(product_new):
    """For a product, returns the P(click) of each profile combination."""
    caracteristicas_producto = features_por_categoria.loc[product_new]
    filas = []
    for combinacion in todas_las_combinaciones:
        # we start from the reference profile and change the swept variables
        perfil = dict(referencia)
        for nombre, valor in zip(nombres_barrido, combinacion):
            perfil[nombre] = valor
        # we build the row: user variables + product variables
        fila = [perfil[c] for c in USER_FEATS] + [caracteristicas_producto[c] for c in PROD_FEATS]
        filas.append(fila)
    X = np.array(filas, dtype=float)
    prob = modelo.predict_proba(X)[:, 1]
    resultado = pd.DataFrame(todas_las_combinaciones, columns=nombres_barrido)
    resultado["p_click"] = prob
    return resultado


def perfiles_top(product_new, n=5):
    resultado = consulta_inversa(product_new).sort_values("p_click", ascending=False).head(n).copy()
    resultado["edad"] = resultado["age_cat"].map(GRUPOS_EDAD)
    resultado["sexo"] = resultado["gender_enc"].map({1: "H", 0: "M"})
    resultado["coche"] = resultado["tiene_coche_enc"].map({1: "sí", 0: "no"})
    return resultado[["edad", "sexo", "ipa_class", "coche", "mun_type", "p_click"]].round(3).reset_index(drop=True)


for producto in ["car_insurance", "funeral_insurance", "smartphone_bundle", "loan"]:
    if producto in features_por_categoria.index:
        print(f"\n--- {producto} --- perfiles con mayor P(click):")
        print(perfiles_top(producto).to_string(index=False))

## 4 · Marginal probabilities by age group

For each product, the mean `P(click)` of each age group (averaged over all combinations).

In [ ]:
PRODUCTOS_DEMO = [c for c in ["car_insurance", "funeral_insurance", "smartphone_bundle", "loan", "travel"]
                  if c in features_por_categoria.index]

# We build a table: rows = age group, columns = product
marginal_edad = {}
for producto in PRODUCTOS_DEMO:
    resultado = consulta_inversa(producto)
    media_por_edad = resultado.groupby("age_cat")["p_click"].mean()
    marginal_edad[producto] = media_por_edad
marginal_edad = pd.DataFrame(marginal_edad)
marginal_edad.index = marginal_edad.index.map(GRUPOS_EDAD)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(marginal_edad.T, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "P(click) media"})
ax.set_title("P(click) media por grupo de edad y producto")
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# How much does age vary within each product? (maximum - minimum)
rango = marginal_edad.max() - marginal_edad.min()
print("Rango de la P(click) por edad en cada producto (máx - mín):")
print(rango.round(3))
print("\nRangos pequeños = el perfil mueve poco P(click); manda el producto (variables sintéticas).")

## Conclusion and decisions

- **The method works:** the M2 model is queried in reverse to see which profiles
  fit each product, and the marginal probabilities per variable are read off.
- **But the profile effect is weak:** the marginals vary little → the product dominates, not the demographics
  (synthetic variables). Consistent with M2 (the signal was in the content).
- For the inverse model to be informative, **real** demographic signal would be needed.